In [46]:
import datetime

import polars as pl
from torch.distributed.tensor import zeros

# Explore

In [47]:
acc = pl.read_csv('../data/HI-Medium_accounts.csv')
trans = pl.read_csv('../data/HI-Medium_Trans.csv').with_columns(
    pl.col("Timestamp").str.strptime(pl.Datetime, "%Y/%m/%d %H:%M")
)

In [48]:
acc.describe()

statistic,Bank Name,Bank ID,Account Number,Entity ID,Entity Name
str,str,f64,str,str,str
"""count""","""2087786""",2.087786e6,"""2087786""","""2087786""","""2087786"""
"""null_count""","""0""",0.0,"""0""","""0""","""0"""
"""mean""",null,632102.369434,null,null,null
"""std""",null,970374.552963,null,null,null
"""min""","""Acme Bancorp""",0.0,"""100428660""","""2AA1CA62570""","""Corporation #1"""
"""25%""",null,39624.0,null,null,null
"""50%""",null,203729.0,null,null,null
"""75%""",null,381212.0,null,null,null
"""max""","""Willows Trust Bank""",3.225455e6,"""8521FC200""","""2AA219D3C80""","""Sole Proprietorship #99999"""


In [49]:
acc.head(10)

Bank Name,Bank ID,Account Number,Entity ID,Entity Name
str,i64,str,str,str
"""China Bank #561""",53267,"""817D00980""","""2AA1F24F180""","""Corporation #183669"""
"""Spain Bank #18657""",316997,"""808BB2280""","""2AA1EEB8540""","""Partnership #193780"""
"""First Bank of Helena""",339367,"""8505ED380""","""2AA206D7790""","""Sole Proprietorship #4078"""
"""Mexico Bank #3367""",3148419,"""8363D4180""","""2AA2001B1A0""","""Partnership #133577"""
"""Switzerland Bank #2372""",3174937,"""842090C80""","""2AA20224CB0""","""Sole Proprietorship #190823"""
"""Italy Bank #19332""",324508,"""80B21CE00""","""2AA1EE12610""","""Corporation #84124"""
"""National Bank of Butte""",368763,"""826283D80""","""2AA204A7FB0""","""Partnership #34044"""
"""Russia Bank #72""",15,"""851500781""","""2AA1D7EC6B0""","""Sole Proprietorship #145838"""
"""Finland Bank #3211""",322280,"""8410D7400""","""2AA1EE17570""","""Sole Proprietorship #214129"""


In [50]:
trans.describe()

statistic,Timestamp,From Bank,From,To Bank,To,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering
str,str,f64,str,f64,str,f64,str,f64,str,str,f64
"""count""","""31898238""",3.1898238e7,"""31898238""",3.1898238e7,"""31898238""",3.1898238e7,"""31898238""",3.1898238e7,"""31898238""","""31898238""",3.1898238e7
"""null_count""","""0""",0.0,"""0""",0.0,"""0""",0.0,"""0""",0.0,"""0""","""0""",0.0
"""mean""","""2022-09-08 15:00:42.879756""",294409.446299,null,409319.754272,null,6.4311e6,null,4.4176e6,null,null,0.001104
"""std""",null,615314.932194,null,654700.254659,null,2.5927e9,null,1.8483e9,null,null,0.033215
"""min""","""2022-09-01 00:00:00""",0.0,"""100428660""",0.0,"""100428660""",0.000001,"""Australian Dollar""",0.000001,"""Australian Dollar""","""ACH""",0.0
"""25%""","""2022-09-03 13:22:00""",2954.0,null,27496.0,null,207.87,null,209.23,null,null,0.0
"""50%""","""2022-09-08 12:14:00""",39024.0,null,146853.0,null,1469.25,null,1471.54,null,null,0.0
"""75%""","""2022-09-13 10:06:00""",215884.0,null,259893.0,null,11835.3,null,11757.81,null,null,0.0
"""max""","""2022-09-28 15:58:00""",3.225455e6,"""8521FC1B0""",3.225455e6,"""8521FC1B0""",8.1586e12,"""Yuan""",8.1586e12,"""Yuan""","""Wire""",1.0


In [51]:
trans.head(10)

Timestamp,From Bank,From,To Bank,To,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering
datetime[μs],i64,str,i64,str,f64,str,f64,str,str,i64
2022-09-01 00:17:00,20,"""800104D70""",20,"""800104D70""",6794.63,"""US Dollar""",6794.63,"""US Dollar""","""Reinvestment""",0
2022-09-01 00:02:00,3196,"""800107150""",3196,"""800107150""",7739.29,"""US Dollar""",7739.29,"""US Dollar""","""Reinvestment""",0
2022-09-01 00:17:00,1208,"""80010E430""",1208,"""80010E430""",1880.23,"""US Dollar""",1880.23,"""US Dollar""","""Reinvestment""",0
2022-09-01 00:03:00,1208,"""80010E650""",20,"""80010E6F0""",7.3966883e7,"""US Dollar""",7.3966883e7,"""US Dollar""","""Cheque""",0
2022-09-01 00:02:00,1208,"""80010E650""",20,"""80010EA30""",4.5868454e7,"""US Dollar""",4.5868454e7,"""US Dollar""","""Cheque""",0
2022-09-01 00:27:00,3203,"""80010EA80""",3203,"""80010EA80""",13284.41,"""US Dollar""",13284.41,"""US Dollar""","""Reinvestment""",0
2022-09-01 00:25:00,20,"""800104D20""",20,"""800104D20""",9.72,"""US Dollar""",9.72,"""US Dollar""","""Reinvestment""",0
2022-09-01 00:09:00,1208,"""80010E430""",1208,"""80010E430""",7.66,"""US Dollar""",7.66,"""US Dollar""","""Reinvestment""",0
2022-09-01 00:09:00,11,"""80010E600""",11,"""80010E600""",16.33,"""US Dollar""",16.33,"""US Dollar""","""Reinvestment""",0


In [52]:
accounts = trans.select(pl.col('From')).vstack(trans.select(pl.col('To')).rename({'To': 'From'})).unique().rename({'From': 'Number'}).with_row_index()
accounts.describe()

statistic,index,Number
str,f64,str
"""count""",2.076999e6,"""2076999"""
"""null_count""",0.0,"""0"""
"""mean""",1.038499e6,null
"""std""",599578.110216,null
"""min""",0.0,"""100428660"""
"""25%""",519250.0,null
"""50%""",1.038499e6,null
"""75%""",1.557749e6,null
"""max""",2.076998e6,"""8521FC1B0"""


In [53]:
accounts.head()

index,Number
u32,str
0,"""841891780"""
1,"""845E0E0F0"""
2,"""80BBE4C40"""
3,"""82A0DA9C0"""
4,"""832517F40"""


In [54]:
trans.filter(pl.col('Amount Paid') != pl.col('Amount Received')).head()

Timestamp,From Bank,From,To Bank,To,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering
datetime[μs],i64,str,i64,str,f64,str,f64,str,str,i64
2022-09-01 00:19:00,4011,"""8032D1A00""",4011,"""8032D1A00""",97.27,"""Euro""",113.98,"""US Dollar""","""ACH""",0
2022-09-01 00:13:00,5991,"""80341B8B0""",5991,"""80341B8B0""",14.26,"""UK Pound""",18.42,"""US Dollar""","""ACH""",0
2022-09-01 00:18:00,0,"""800417500""",0,"""800417500""",42.41,"""Euro""",49.69,"""US Dollar""","""ACH""",0
2022-09-01 00:11:00,3504,"""800492D00""",3504,"""800492D00""",6707.83,"""US Dollar""",5724.46,"""Euro""","""ACH""",0
2022-09-01 00:29:00,1488,"""800C948C0""",1488,"""800C948C0""",42.71,"""Euro""",50.05,"""US Dollar""","""ACH""",0


In [55]:
import gc

del accounts
del acc
del trans

gc.collect()

586

# Medium

In [56]:
trans = (
    pl.read_csv('../data/HI-Medium_Trans.csv')
    .with_columns(
        pl.col("Timestamp").str.strptime(pl.Datetime, "%Y/%m/%d %H:%M")
    )
    .sort('Timestamp')
)

# Cutoff for time leakage

In [57]:
import datetime

## Time splits for row proportion splitting

In [58]:
zero = trans['Timestamp'].min()
sixty = trans['Timestamp'].quantile(0.6)
eighty = trans['Timestamp'].quantile(0.8)
hundred = trans['Timestamp'].max()

print(zero, sixty, eighty, hundred)

2022-09-01 00:00:00 2022-09-09 20:43:00 2022-09-14 05:46:00 2022-09-28 15:58:00


In [59]:
train_row =  trans.filter(pl.col('Timestamp') <= sixty)
val_row = trans.filter(pl.col('Timestamp') <= eighty).filter(pl.col('Timestamp') > sixty)
test_row = trans.filter(pl.col('Timestamp') > eighty)

## Time splits for day splitting

In [60]:
diff = hundred - zero
days = diff.days

In [61]:
sixty = zero + datetime.timedelta(days=days * 0.6)
eighty = zero + datetime.timedelta(days=days * 0.8)
hundred = zero + datetime.timedelta(days=days)

print(zero, sixty, eighty, hundred)

2022-09-01 00:00:00 2022-09-17 04:48:00 2022-09-22 14:24:00 2022-09-28 00:00:00


In [62]:
train =  trans.filter(pl.col('Timestamp') <= sixty)
val = trans.filter(pl.col('Timestamp') <= eighty).filter(pl.col('Timestamp') > sixty)
test = trans.filter(pl.col('Timestamp') > eighty)

# Currency behavior

In [63]:
trans.group_by(pl.col('From')).agg(pl.len()).sort(pl.col('len'), descending=True).limit(10)

From,len
str,u32
"""100428660""",1076979
"""1004286A8""",678929
"""1004286F0""",208695
"""1004289C0""",132783
"""100428858""",102358
"""100428810""",96007
"""1004287C8""",90963
"""1004288A0""",87870
"""100428738""",80578


In [64]:
trans.filter(pl.col('From').is_in(["100428660", "1004286A8", "1004286F0"])).group_by(['From', 'Receiving Currency']).len()

From,Receiving Currency,len
str,str,u32
"""1004286A8""","""Euro""",678929
"""1004286F0""","""Yuan""",208695
"""100428660""","""US Dollar""",1076979


In [65]:
trans.filter(pl.col('From').is_in(["100428660", "1004286A8", "1004286F0"])).group_by(['From', 'Payment Currency']).len()

From,Payment Currency,len
str,str,u32
"""100428660""","""US Dollar""",1076979
"""1004286A8""","""Euro""",678929
"""1004286F0""","""Yuan""",208695
